# 02 — Fine-Tuning

This notebook documents Phase 2: fine-tuning `gpt-oss-20b` on the ChatTLA corpus.

> **Running this notebook:**
> Execute the cells in order for dataset preparation and smoke tests.  Full
> training is **not** performed inside the notebook; launch it from a terminal
> using the command below (see Step 3 for details):
> ```bash
> CUDA_VISIBLE_DEVICES=1 python -m src.training.train
> ```
> After the run finishes you can view metrics in the MLflow UI and then switch
> to the evaluation notebook (`03_evaluation.ipynb`) or run `python -m
> src.inference.benchmark` directly.

## Setup Summary

| Component | Choice | Reason |
|-----------|--------|--------|
| Base model | `openai/gpt-oss-20b` | 20B total (3.6B active MoE), fits in 49 GB VRAM, fine-tunable on consumer hardware per model card |
| GPU | Quadro RTX 8000, device 1 | GPU 0 in use; GPU 1 is idle at 22W |
| Quantisation | `Mxfp4Config(dequantize=True)` → BF16 | Load MXFP4 weights, de-quant to BF16 for gradient precision |
| PEFT | LoRA rank 8, `all-linear` + `target_parameters` for MoE experts | Architecture-specific: MoE expert layers (blocks 7, 15, 23) must be explicitly targeted |
| Trainer | `SFTTrainer` (TRL) | Native harmony `messages` support, gradient checkpointing |
| Prompt format | gpt-oss harmony (required) | Model card: *"will not work correctly"* without it |

## Primary Metric

> **`tlc_clean_rate`** — fraction of eval-set generated specs that TLC model-checks with no violations

Target: > 0.70 at end of training.  Tracked by `TLCEvalCallback` at every `eval_steps=100`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path('.').resolve()

## Step 1: Build Training Dataset

Generates `data/processed/train.jsonl` and `data/processed/eval.jsonl` from `data/validated/combined.jsonl`.

Four task types are created:
- **spec_generation** (~50%): NL description → full spec
- **spec_completion** (~20%): partial spec → complete spec
- **invariant_gen** (~20%): full spec → invariant operators
- **bug_fix** (~10%): buggy spec + TLC error → fixed spec

Each example is pre-formatted in the gpt-oss harmony message structure.

In [ ]:
from src.training.dataset_builder import build
n_train, n_eval = build()
print(f'Training examples: {n_train}')
print(f'Eval examples:     {n_eval}')

In [ ]:
# Inspect what a training example looks like
train_jsonl = REPO_ROOT / 'data' / 'processed' / 'train.jsonl'
if train_jsonl.exists():
    first = json.loads(train_jsonl.open().readline())
    for msg in first['messages']:
        role    = msg['role']
        channel = msg.get('channel', '')
        content = msg['content'][:200]
        print(f"[{role}/{channel}] {content}")
        print('---')

## Step 2: Smoke Test Training Setup

Runs 10 training steps with 10 examples to validate:
- Model loads correctly with `Mxfp4Config(dequantize=True)` → BF16
- LoRA config targets the MoE expert layers without error
- `CUDA_VISIBLE_DEVICES=1` correctly pins to GPU 1
- Gradient flow is non-zero
- `TLCEvalCallback` runs without crashing

**This is not training** — it exits after 10 steps. Look for `loss` decreasing monotonically.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'src.training.train', '--smoke-test'],
    env={**__import__('os').environ, 'CUDA_VISIBLE_DEVICES': '1'},
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

## Step 3: Full Training Run

Launch the full training run from terminal (not from notebook — it takes hours):

```bash
CUDA_VISIBLE_DEVICES=1 python -m src.training.train
```

Monitor via MLflow UI:
```bash
mlflow ui --port 5000
```

**Key metrics to watch**:
- `train/loss` decreasing
- `tlc/sany_parse_rate` > 0.90 by step 500
- `tlc/tlc_clean_rate` > 0.70 by end of training

In [ ]:
# Load and plot training metrics from MLflow (after training completes)
import mlflow

mlflow.set_experiment('ChatTLA-gpt-oss-20b')
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(experiment_ids=['1'], order_by=['start_time DESC'])

if runs:
    run = runs[0]
    run_id = run.info.run_id
    print(f'Latest run: {run_id}')

    sany_hist  = client.get_metric_history(run_id, 'tlc/sany_parse_rate')
    tlc_hist   = client.get_metric_history(run_id, 'tlc/tlc_clean_rate')
    loss_hist  = client.get_metric_history(run_id, 'train/loss')

    if sany_hist:
        steps  = [m.step for m in sany_hist]
        sany_v = [m.value for m in sany_hist]
        tlc_v  = [m.value for m in tlc_hist] if tlc_hist else []

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(steps, sany_v, label='SANY parse rate', marker='o')
        if tlc_v:
            axes[0].plot(steps, tlc_v, label='TLC clean rate', marker='s')
        axes[0].axhline(0.90, ls='--', color='gray', lw=0.8, label='SANY target')
        axes[0].axhline(0.70, ls='--', color='red',  lw=0.8, label='TLC target')
        axes[0].set_title('TLC Validation Metrics over Training')
        axes[0].set_xlabel('Training step')
        axes[0].set_ylabel('Rate')
        axes[0].legend()

        if loss_hist:
            loss_steps = [m.step for m in loss_hist]
            loss_v = [m.value for m in loss_hist]
            axes[1].plot(loss_steps, loss_v, color='purple')
            axes[1].set_title('Training Loss')
            axes[1].set_xlabel('Step')
            axes[1].set_ylabel('Loss')

        plt.tight_layout()
        plt.savefig(REPO_ROOT / 'outputs' / 'training_metrics.png', dpi=150)
        plt.show()
    else:
        print('No TLC metrics logged yet — training still running or not started.')
else:
    print('No MLflow runs found. Start training first.')